# WakeMarch — API test and demonstration

A runnable, data-free notebook for the **installed** WakeMarch package. Use a kernel from the environment where WakeMarch is installed, then **Restart Kernel and Run All Cells**. No data downloads or source-path modifications are needed.

This notebook checks the public API, shows a live progress bar, plots the wake, and verifies that progress reporting does not alter the solution. Numerical regression against the frozen research reference cases is part of the release checks.

To set up an environment: `python -m pip install "wakemarch[notebooks]"` (or `".[notebooks]"` from a source checkout), then start `python -m jupyterlab` from that environment.

This example deliberately uses the direct, fixed-CT wake-only interface. It requires no CP or turbine curve. For fixed-coefficient power estimates and curve/preset operating points, see `wakemarch_turbines.ipynb`; those power assumptions are separate from this wake solve.

In [ ]:
import sys
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
import wakemarch
from wakemarch import Numerics, WakeModel

print(f"WakeMarch {wakemarch.__version__}")
print(f"Python: {sys.executable}")
print(f"Installed package: {wakemarch.__file__}")
assert "site-packages" in Path(wakemarch.__file__).parts, "Install the package before running this notebook."
%matplotlib inline

## 1. Choose a turbine and numerical resolution

Lengths are in metres; wind speed in m/s; TI is a **fraction**; yaw is in degrees. Positive yaw deflects towards **negative y**. Sampled z is height above ground.

The demonstration explicitly uses D/20 for a short interactive run. The package default remains D/30. Change these inputs and rerun the cells below.

In [ ]:
model = WakeModel(
    D=126.0, H=90.0, U=8.0, CT=0.8,
    yaw_deg=25.0, TI=0.01, shear_alpha=0.0,
    numerics=Numerics(h_D=1 / 20),
)
max_dist_D = 10.0
model

## 2. Solve with live progress

Progress is on by default. It includes the extra downstream marching needed for yaw reconstruction, so its total distance exceeds the requested 10D. The bar shows preparation, marching and completion. `progress=False` runs silently.

In [ ]:
result = model.solve(max_dist_D=max_dist_D)
assert np.isfinite(result.marching_deficit).all()
assert np.isfinite(result.streamwise_vorticity).all()
print(f"Stored history: {result.marching_deficit.shape} (x_s, z_relative, y)")
print(f"Last stored marching plane: {result.x[-1] / model.D:.2f} D")

## 3. Distinguish velocity from deficit

At uniform inflow, streamwise velocity + deficit equals U. With shear it equals the power-law background at the sampled height. Scalar, list and broadcast-array queries are supported.

In [ ]:
x = 5 * model.D
hub_deficit = result.sample_deficit(x, 0, model.H)
hub_velocity = result.sample_streamwise_velocity(x, 0, model.H)
assert isinstance(hub_deficit, float)
np.testing.assert_allclose(hub_velocity + hub_deficit, model.U)
print(f"At 5D on the turbine axis: deficit={hub_deficit:.3f} m/s; velocity={hub_velocity:.3f} m/s")

heights = np.array([model.H - 0.25 * model.D, model.H, model.H + 0.25 * model.D])
velocities = result.sample_streamwise_velocity(x, 0, heights)
deficits = result.sample_deficit(x, 0, heights)
expected_background = model.U * (np.maximum(heights, 0.05 * model.H) / model.H) ** model.shear_alpha
np.testing.assert_allclose(velocities + deficits, expected_background)
upstream = result.sample_deficit(-model.D, 0, model.H)  # induction zone of the rotor (near field)
assert 0 < upstream < 0.05 * model.U
print("Scalar, array, background and upstream checks passed.")

## 4. Plot the reconstructed wake

These plots use the public sampler, including the yaw-coordinate reconstruction and the near-field display of the rotor. The raw marching histories are not physical x-y/z fields.

In [ ]:
x_D = np.linspace(0, max_dist_D, 141)
y_D = np.linspace(-1.5, 1.5, 101)
z_D = np.linspace(0, model.H / model.D + 1.3, 101)
xx, yy = np.meshgrid(x_D * model.D, y_D * model.D)
hub_plane = result.sample_deficit(xx, yy, model.H) / model.U
yy_cross, zz_cross = np.meshgrid(y_D * model.D, z_D * model.D)
cross_plane = result.sample_deficit(5 * model.D, yy_cross, zz_cross) / model.U
assert hub_plane.shape == xx.shape and cross_plane.shape == yy_cross.shape

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
vmax = max(hub_plane.max(), cross_plane.max())
axes[0].pcolormesh(x_D, y_D, hub_plane, shading="auto", cmap="viridis", vmin=0, vmax=vmax)
axes[0].set(xlabel="x / D", ylabel="y / D", title="Hub-height deficit")
image = axes[1].pcolormesh(y_D, z_D, cross_plane, shading="auto", cmap="viridis", vmin=0, vmax=vmax)
axes[1].set(xlabel="y / D", ylabel="Height above ground / D", title="Cross-plane deficit at x = 5D", aspect="equal")
axes[1].axhline(model.H / model.D, color="white", lw=0.7, ls="--")
fig.colorbar(image, ax=axes, label="Deficit / hub wind speed")
plt.show()

## 5. Bounds and input validation

Outside stored downstream support, sampling raises an error by default. Explicit `bounds="zero"` extends with zero deficit (background velocity); it does not compute a distant wake. Input checks guard computation, not the entire experimentally validated operating range.

In [ ]:
try:
    result.sample_deficit(1e6 * model.D, 0, model.H)
except ValueError as exc:
    print("Expected bounds error:", exc)
else:
    raise AssertionError("Outside-support sampling should fail by default")
assert result.sample_deficit(1e6 * model.D, 0, model.H, bounds="zero") == 0
try:
    replace(model, TI=6)
except ValueError:
    print("TI=6 correctly rejected: use 0.06 for 6%.")
else:
    raise AssertionError("Invalid TI was accepted")

## 6. Test progress equivalence and a custom callback

A small coarse case runs twice, once silently and once with a callback. All three stored field histories must be **exactly identical**. The callback receives `(completed_D, total_D)` and can be used with an external progress library.

In [ ]:
small = replace(model, numerics=Numerics(h_D=1 / 10))
quiet = small.solve(max_dist_D=1.0, progress=False)
events = []
observed = small.solve(max_dist_D=1.0, progress=lambda done, total: events.append((done, total)))
for field in ("marching_deficit", "streamwise_vorticity", "streamwise_velocity_derivative"):
    np.testing.assert_array_equal(getattr(quiet, field), getattr(observed, field))
assert events[0][0] == 0
assert events[-1][0] == events[-1][1]
assert all(a[0] <= b[0] for a, b in zip(events, events[1:]))
assert events[-1][1] > 1.0
assert not observed.marching_deficit.flags.writeable
print(f"Progress checks passed: {len(events)} updates; all saved fields identical.")

## 7. Compare yaw angles

These are single-turbine demonstrations, not wind-farm or power predictions. Reduce the angle list for a faster run.

In [ ]:
yaw_profiles = {}
y_line = np.linspace(-1.5 * model.D, 1.5 * model.D, 121)
for yaw in (0, 15, 30):
    print(f"Yaw = {yaw}°")
    yaw_model = replace(model, yaw_deg=yaw)
    yaw_result = yaw_model.solve(max_dist_D=6)
    yaw_profiles[yaw] = yaw_result.sample_deficit(5 * model.D, y_line, model.H) / model.U

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for yaw, profile in yaw_profiles.items():
    ax.plot(y_line / model.D, profile, label=f"{yaw}°")
ax.set(xlabel="y / D", ylabel="Deficit / hub wind speed", title="Hub-height profiles at 5D")
ax.legend(title="Yaw")
ax.grid(alpha=0.2)
plt.show()
print("All notebook API checks passed.")

## Scope

This notebook checks API behaviour and visualizes the accepted engineering model. It is not new observational validation. Known limitations include near-wake reconstruction, high-TI mixing and high-yaw far-wake behaviour. Neither exact circulation conservation nor exact discrete force consistency is established by these tests.

WakeMarch models a single turbine; it is not a wind-farm solver.